In [ ]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"

Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [ ]:
import sys
import importlib

# Add the Silver layer directory to sys.path BEFORE importing
silver_dir = "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver"
if silver_dir not in sys.path:
    sys.path.insert(0, silver_dir)

# Now import (and reload to pick up edits)
import utils_silver
from utils_silver import *

importlib.reload(utils_silver)

paths_bronze, paths_silver, paths_gold = utils_silver.build_paths(storage_account1)
print(sorted(paths_silver.keys()))
print("Imported utils_silver from", utils_silver.__file__)

✅ utils_silver.py loaded
['_metrics', '_quarantine', '_ref_dim_channel', '_ref_dim_product_line', '_reference', 'claims', 'members', 'policies', 'providers']
Imported utils_silver from /Users/manojrammopati/Public/Projects/bupa_insurance_project/02_Silver_Layer/utils_silver.py


# Cell 1 – Imports & config

In [5]:
from pyspark.sql import functions as F

# Databases
DB_SILVER = "bupa_silver"
DB_GOLD   = "bupa_gold"

# Storage
GOLD_BASE = "abfss://golddata@clientdatastorage.dfs.core.windows.net"

# Paths for this mart
DM_MEMBER_VALUE_PATH = f"{GOLD_BASE}/dm_member_value"

print("DB_GOLD:", DB_GOLD)
print("GOLD_BASE:", GOLD_BASE)
print("DM_MEMBER_VALUE_PATH:", DM_MEMBER_VALUE_PATH)


DB_GOLD: bupa_gold
GOLD_BASE: abfss://golddata@clientdatastorage.dfs.core.windows.net
DM_MEMBER_VALUE_PATH: abfss://golddata@clientdatastorage.dfs.core.windows.net/dm_member_value


# Cell 2 – Load Gold fact tables

We’ll use members + policies + claims together.

In [10]:
# Replace with your actual storage account and path
fact_policies = spark.read.format("delta").load("abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_policies")
fact_members  = spark.read.format("delta").load("abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_members")
fact_claims   = spark.read.format("delta").load("abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_claims")

In [13]:
DB_GOLD = "bupa_gold"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_GOLD}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.fact_policies
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_policies'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.fact_members
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_members'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.fact_claims
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_claims'
""")

DataFrame[]

In [14]:
# Fact tables in Gold
fact_members  = spark.table(f"{DB_GOLD}.fact_members")
fact_policies = spark.table(f"{DB_GOLD}.fact_policies")
fact_claims   = spark.table(f"{DB_GOLD}.fact_claims")

print("fact_members:",  fact_members.count())
print("fact_policies:", fact_policies.count())
print("fact_claims:",   fact_claims.count())

fact_members.show(5, truncate=False)
fact_policies.show(5, truncate=False)
fact_claims.show(5, truncate=False)


25/12/06 20:17:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


fact_members: 381109


fact_policies: 381109
fact_claims: 558211


+------------+----------+---+------+------------------+-----------+---------------+-----------------+------+--------+----------+-------------+------------+-------------+----------------+--------------+------------+------------+
|Member_ID   |Policy_ID |Age|Gender|BMI               |Smoker_Flag|Chronic_Disease|Employment_Status|Region|Age_Band|BMI_Band  |Smoker_Status|Chronic_Flag|Chronic_Group|Employment_Group|Region_Group  |dq_age_valid|dq_bmi_valid|
+------------+----------+---+------+------------------+-----------+---------------+-----------------+------+--------+----------+-------------+------------+-------------+----------------+--------------+------------+------------+
|MEM_00000008|P_00000008|56 |F     |23.140940890221643|Y          |Asthma         |Student          |280   |45–59   |Normal    |Smoker       |1           |Asthma       |Student         |Region_250-349|1           |1           |
|MEM_00001714|P_00001714|71 |M     |27.963719840043765|N          |None           |Emplo

+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+-----------+-----------------+-------------+---------------+--------------+-----------------+----------------+--------------------------+----------------------------+----------------------------------------------------------------+
|Policy_ID |Customer_ID |Product_Line|Channel|Sum_Insured_GBP   |Annual_Premium_GBP|Policy_Start_Date|Policy_End_Date|Policy_Duration_Days|Renewal_Offered_Flag|Renewal_Accepted_Flag|Renewal_Conversion|Tenure_Band|Premium_Band     |Discount_Band|Renewal_Outcome|dq_money_valid|dq_discount_valid|dq_renewal_valid|created_at                |source_system               |record_hash                                                     |
+----------+------------+------------+-------+------------------+------------------+-----------------+---------------+----------------

+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+-------------------+--------------+----------------------+--------------------+--------------+-------------+
|Claim_ID |Provider_ID|Member_Key|Date_Reported|Date_Settled|Payout_GBP|Claim_Amount_GBP  |Fraud_Label|Claim_Type|Claim_Status|Provider_Fraud_Flag|Days_To_Settle|Payout_to_Amount_Ratio|High_Cost_Claim_Flag|dq_money_valid|dq_date_valid|
+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+-------------------+--------------+----------------------+--------------------+--------------+-------------+
|CLM110013|PRV51790   |BENE15435 |NULL         |NULL        |200.0     |244.07511199775644|0          |Travel    |Settled     |0                  |NULL          |0.8194198841618822    |0                   |1             |1            |
|CLM110020|PRV53679   |BENE53030 |NULL         |NULL    

# Cell 3 – Build base member–policy–claims view

Goal: one row per member + policy, with aggregated claim metrics.

In [15]:
# 1) Join members to policies on Policy_ID
mem_pol = (
    fact_members.alias("m")
    .join(
        fact_policies.alias("p"),
        on="Policy_ID",
        how="left"
    )
)

print("Joined members → policies:", mem_pol.count())
mem_pol.select("Member_ID", "Policy_ID", "Product_Line", "Channel", "Age", "Gender", "Region").show(5, truncate=False)


Joined members → policies: 381109


+------------+----------+------------+-------+---+------+------+
|Member_ID   |Policy_ID |Product_Line|Channel|Age|Gender|Region|
+------------+----------+------------+-------+---+------+------+
|MEM_00000008|P_00000008|Health      |Partner|56 |F     |280   |
|MEM_00001714|P_00001714|Health      |Partner|71 |M     |360   |
|MEM_00001746|P_00001746|Dental      |Partner|75 |M     |80    |
|MEM_00001967|P_00001967|Health      |Partner|37 |F     |280   |
|MEM_00001995|P_00001995|Accident    |Partner|66 |M     |360   |
+------------+----------+------------+-------+---+------+------+
only showing top 5 rows



# Now aggregate claims by Member_Key / Policy_ID.
(Recall: in claims we used Member_Key as the join key.

In [16]:
# 2) Aggregate claims at member-policy level
claims_agg = (
    fact_claims
    .groupBy("Member_Key")  # you could add Policy_ID if available later
    .agg(
        F.count("*").alias("Total_Claims"),
        F.sum("Payout_GBP").alias("Total_Payout_GBP"),
        F.sum("Claim_Amount_GBP").alias("Total_Claim_Amount_GBP"),
        F.max("Fraud_Label").alias("Has_Fraud_Claim"),
        F.max("Date_Reported").alias("Last_Claim_Date")
    )
)

print("Claims aggregated at Member_Key level:", claims_agg.count())
claims_agg.show(5, truncate=False)


Claims aggregated at Member_Key level: 138556


+----------+------------+----------------+----------------------+---------------+---------------+
|Member_Key|Total_Claims|Total_Payout_GBP|Total_Claim_Amount_GBP|Has_Fraud_Claim|Last_Claim_Date|
+----------+------------+----------------+----------------------+---------------+---------------+
|BENE57558 |2           |2030.0          |2060.307316351115     |0              |NULL           |
|BENE41564 |11          |3430.0          |4416.152942704686     |1              |NULL           |
|BENE48136 |4           |690.0           |958.5110135342991     |0              |NULL           |
|BENE64207 |6           |11570.0         |17248.67537137078     |1              |NULL           |
|BENE68523 |17          |23450.0         |30570.073097939738    |0              |NULL           |
+----------+------------+----------------+----------------------+---------------+---------------+
only showing top 5 rows



# Now join aggregated claims into the member–policy frame:

In [17]:
# 3) Join claims into member + policy
fact_base = (
    mem_pol
    .join(
        claims_agg.alias("c"),
        mem_pol["Member_ID"] == F.col("c.Member_Key"),
        how="left"
    )
    .drop("Member_Key")  # from c; Member_ID is your canonical key
)

print("fact_base rowcount:", fact_base.count())
fact_base.select(
    "Member_ID", "Policy_ID", "Total_Claims", "Total_Payout_GBP", "Has_Fraud_Claim"
).show(5, truncate=False)


fact_base rowcount: 381109


+------------+----------+------------+----------------+---------------+
|Member_ID   |Policy_ID |Total_Claims|Total_Payout_GBP|Has_Fraud_Claim|
+------------+----------+------------+----------------+---------------+
|MEM_00000008|P_00000008|NULL        |NULL            |NULL           |
|MEM_00001714|P_00001714|NULL        |NULL            |NULL           |
|MEM_00001746|P_00001746|NULL        |NULL            |NULL           |
|MEM_00001967|P_00001967|NULL        |NULL            |NULL           |
|MEM_00001995|P_00001995|NULL        |NULL            |NULL           |
+------------+----------+------------+----------------+---------------+
only showing top 5 rows



# Cell 4 – Derived member value / risk features

Here we compute business-friendly features:

Claim frequency bands

Total payout bands

Engagement category (claiming vs non-claiming)

Risk tag (very simple rules for demo)

In [18]:
fact_enriched = (
    fact_base
    # Claim frequency band
    .withColumn(
        "Claim_Frequency_Band",
        F.when(F.col("Total_Claims").isNull() | (F.col("Total_Claims") == 0), "No claims")
         .when(F.col("Total_Claims") == 1, "1 claim")
         .when(F.col("Total_Claims") <= 3, "2–3 claims")
         .otherwise("4+ claims")
    )
    # Total payout band
    .withColumn(
        "Total_Payout_Band",
        F.when(F.col("Total_Payout_GBP").isNull() | (F.col("Total_Payout_GBP") == 0), "No payout")
         .when(F.col("Total_Payout_GBP") < 500,  "< 500")
         .when(F.col("Total_Payout_GBP") < 2000, "500–2k")
         .when(F.col("Total_Payout_GBP") < 5000, "2k–5k")
         .otherwise("5k+")
    )
    # Engagement type: based on claims
    .withColumn(
        "Engagement_Type",
        F.when(F.col("Total_Claims").isNull() | (F.col("Total_Claims") == 0), "No-touch")
         .when(F.col("Total_Claims") <= 2, "Low-touch")
         .otherwise("High-touch")
    )
    # Simple risk tag (very explainable)
    .withColumn(
        "Risk_Tag",
        F.when(F.col("Has_Fraud_Claim") == 1, "High risk – fraud history")
         .when(F.col("Total_Claims") >= 4, "High risk – heavy user")
         .when(F.col("Total_Claims").between(2, 3), "Medium risk")
         .otherwise("Low risk")
    )
)

fact_enriched.select(
    "Member_ID", "Policy_ID", "Total_Claims", "Total_Payout_GBP",
    "Claim_Frequency_Band", "Total_Payout_Band", "Engagement_Type", "Risk_Tag"
).show(10, truncate=False)

fact_enriched.printSchema()


+------------+----------+------------+----------------+--------------------+-----------------+---------------+--------+
|Member_ID   |Policy_ID |Total_Claims|Total_Payout_GBP|Claim_Frequency_Band|Total_Payout_Band|Engagement_Type|Risk_Tag|
+------------+----------+------------+----------------+--------------------+-----------------+---------------+--------+
|MEM_00000008|P_00000008|NULL        |NULL            |No claims           |No payout        |No-touch       |Low risk|
|MEM_00001714|P_00001714|NULL        |NULL            |No claims           |No payout        |No-touch       |Low risk|
|MEM_00001746|P_00001746|NULL        |NULL            |No claims           |No payout        |No-touch       |Low risk|
|MEM_00001967|P_00001967|NULL        |NULL            |No claims           |No payout        |No-touch       |Low risk|
|MEM_00001995|P_00001995|NULL        |NULL            |No claims           |No payout        |No-touch       |Low risk|
|MEM_00002251|P_00002251|NULL        |NU

# Cell 5 – Write dm_member_value to Gold (Delta + table)

In [19]:
# 1) Write as Delta files
(
    fact_enriched
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DM_MEMBER_VALUE_PATH)
)
print("✅ dm_member_value written to:", DM_MEMBER_VALUE_PATH)

# 2) Register as a Gold table
spark.sql("CREATE DATABASE IF NOT EXISTS bupa_gold")
spark.sql("DROP TABLE IF EXISTS bupa_gold.dm_member_value")

spark.sql(f"""
CREATE TABLE bupa_gold.dm_member_value
USING DELTA
LOCATION '{DM_MEMBER_VALUE_PATH}'
""")

print("✅ Registered table: bupa_gold.dm_member_value")

# 3) Sanity check
spark.table("bupa_gold.dm_member_value").show(10, truncate=False)


25/12/06 20:21:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


✅ dm_member_value written to: abfss://golddata@clientdatastorage.dfs.core.windows.net/dm_member_value
✅ Registered table: bupa_gold.dm_member_value


+----------+------------+---+------+------------------+-----------+---------------+-----------------+------+--------+----------+-------------+------------+-------------+----------------+--------------+------------+------------+------------+------------+-------+------------------+------------------+-----------------+---------------+--------------------+--------------------+---------------------+------------------+-----------+-----------------+-------------+---------------+--------------+-----------------+----------------+--------------------------+----------------------------+----------------------------------------------------------------+------------+----------------+----------------------+---------------+---------------+--------------------+-----------------+---------------+--------+
|Policy_ID |Member_ID   |Age|Gender|BMI               |Smoker_Flag|Chronic_Disease|Employment_Status|Region|Age_Band|BMI_Band  |Smoker_Status|Chronic_Flag|Chronic_Group|Employment_Group|Region_Group  |dq

# Cell A – Basic profile / row counts

In [20]:
dm = spark.table("bupa_gold.dm_member_value")

print("Rowcount (dm_member_value):", dm.count())
print("Distinct members:", dm.select("Member_ID").distinct().count())
print("Distinct policies:", dm.select("Policy_ID").distinct().count())

dm.select(
    "Member_ID", "Policy_ID", "Age", "Gender", "Region",
    "Total_Claims", "Total_Payout_GBP", "Risk_Tag"
).show(10, truncate=False)


Rowcount (dm_member_value): 381109


Distinct members: 381109


Distinct policies: 381109
+------------+----------+---+------+------+------------+----------------+--------+
|Member_ID   |Policy_ID |Age|Gender|Region|Total_Claims|Total_Payout_GBP|Risk_Tag|
+------------+----------+---+------+------+------------+----------------+--------+
|MEM_00000002|P_00000002|76 |M     |30    |NULL        |NULL            |Low risk|
|MEM_00000017|P_00000017|25 |F     |450   |NULL        |NULL            |Low risk|
|MEM_00000022|P_00000022|49 |M     |280   |NULL        |NULL            |Low risk|
|MEM_00000027|P_00000027|51 |F     |280   |NULL        |NULL            |Low risk|
|MEM_00000038|P_00000038|25 |F     |280   |NULL        |NULL            |Low risk|
|MEM_00000044|P_00000044|38 |F     |350   |NULL        |NULL            |Low risk|
|MEM_00000047|P_00000047|29 |F     |150   |NULL        |NULL            |Low risk|
|MEM_00000057|P_00000057|39 |M     |280   |NULL        |NULL            |Low risk|
|MEM_00000067|P_00000067|41 |M     |280   |NULL        |NULL 

# Cell B – Distribution checks (age, risk, engagement)

In [21]:
from pyspark.sql import functions as F

# Age bands for a quick check
dm_age = dm.withColumn(
    "Age_Band",
    F.when(F.col("Age") < 25, "<25")
     .when(F.col("Age") < 40, "25–39")
     .when(F.col("Age") < 55, "40–54")
     .when(F.col("Age") < 70, "55–69")
     .otherwise("70+")
)

print("▶ Age vs Risk_Tag")
(
    dm_age
    .groupBy("Age_Band", "Risk_Tag")
    .agg(F.countDistinct("Member_ID").alias("Members"))
    .orderBy("Age_Band", "Risk_Tag")
    .show(50, truncate=False)
)

print("▶ Engagement_Type distribution")
(
    dm.groupBy("Engagement_Type")
      .agg(F.countDistinct("Member_ID").alias("Members"))
      .orderBy("Members", ascending=False)
      .show(truncate=False)
)


▶ Age vs Risk_Tag


+--------+--------+-------+
|Age_Band|Risk_Tag|Members|
+--------+--------+-------+
|25–39   |Low risk|115587 |
|40–54   |Low risk|104942 |
|55–69   |Low risk|48961  |
|70+     |Low risk|17750  |
|<25     |Low risk|93869  |
+--------+--------+-------+

▶ Engagement_Type distribution


+---------------+-------+
|Engagement_Type|Members|
+---------------+-------+
|No-touch       |381109 |
+---------------+-------+



# Cell C – Product / channel view at member level

Nice for dashboards: “which member segments are driving claims by product/channel?”

In [22]:
print("▶ Member value by Product_Line & Channel")

(
    dm.groupBy("Product_Line", "Channel")
      .agg(
          F.countDistinct("Member_ID").alias("Members"),
          F.sum("Total_Claims").alias("Total_Claims"),
          F.sum("Total_Payout_GBP").alias("Total_Payout_GBP")
      )
      .orderBy(F.col("Total_Payout_GBP").desc())
      .show(50, truncate=False)
)


▶ Member value by Product_Line & Channel


+------------+-------+-------+------------+----------------+
|Product_Line|Channel|Members|Total_Claims|Total_Payout_GBP|
+------------+-------+-------+------------+----------------+
|Health      |Online |635    |NULL        |NULL            |
|Health      |Agent  |311    |NULL        |NULL            |
|Accident    |Agent  |14     |NULL        |NULL            |
|Motor       |Agent  |65     |NULL        |NULL            |
|Accident    |Broker |6      |NULL        |NULL            |
|Dental      |Broker |157    |NULL        |NULL            |
|Health      |Broker |295    |NULL        |NULL            |
|Dental      |Agent  |137    |NULL        |NULL            |
|Health      |Partner|227352 |NULL        |NULL            |
|Accident    |Partner|6427   |NULL        |NULL            |
|Motor       |Online |108    |NULL        |NULL            |
|Travel      |Partner|397    |NULL        |NULL            |
|Dental      |Partner|106367 |NULL        |NULL            |
|Motor       |Broker |53

# Cell D – Suggested Gold KPIs for dashboards (Members & related)

This cell doesn’t run aggregations only; it’s mainly a printed checklist of KPIs to talk about in interviews / use in Power BI.

In [23]:
kpis = [
    "1) Member Count by Risk_Tag (Low / Medium / High)",
    "2) Member Count by Engagement_Type (No-touch / Low-touch / High-touch)",
    "3) Average Total_Payout_GBP per member by Product_Line",
    "4) Distribution of Claim_Frequency_Band by Age_Band and Region",
    "5) % of members with Has_Fraud_Claim = 1",
    "6) Top Regions by Total_Payout_GBP (claim heavy regions)",
    "7) Cross-tab: Risk_Tag vs Renewal_Conversion (from policies)",
    "8) Members with zero claims but high Sum_Insured_GBP (profitable, low-use customers)",
]

print("📊 Recommended Gold KPIs built on dm_member_value:")
for k in kpis:
    print("  -", k)

print("""
These complement:
  • Policy-level KPIs from dm_policy_retention (renewal, discount, duration)
  • Claims-level KPIs from fact_claims / claims mart (frequency, severity, fraud)
Together they give an end-to-end view: policy → member → claims.
""")


📊 Recommended Gold KPIs built on dm_member_value:
  - 1) Member Count by Risk_Tag (Low / Medium / High)
  - 2) Member Count by Engagement_Type (No-touch / Low-touch / High-touch)
  - 3) Average Total_Payout_GBP per member by Product_Line
  - 4) Distribution of Claim_Frequency_Band by Age_Band and Region
  - 5) % of members with Has_Fraud_Claim = 1
  - 6) Top Regions by Total_Payout_GBP (claim heavy regions)
  - 7) Cross-tab: Risk_Tag vs Renewal_Conversion (from policies)
  - 8) Members with zero claims but high Sum_Insured_GBP (profitable, low-use customers)

These complement:
  • Policy-level KPIs from dm_policy_retention (renewal, discount, duration)
  • Claims-level KPIs from fact_claims / claims mart (frequency, severity, fraud)
Together they give an end-to-end view: policy → member → claims.

